# C3E — D1 scores and audit notebook

Supplementary material for *The ROC–PR Divergence and Threshold Transferability: A Cost-Calibrated
Evaluation Protocol for Fraud Detection*.

This notebook fits the six D1 (creditcard) configurations at seed 42 and saves their scores
(Section 1), then recomputes from these scores, and from `results/c3e_all_seeds_raw.csv`, the
quantities of Sections 6.2 and 6.3 and Appendix B of the paper (Tables 8, 9 and 10). It writes into a
timestamped folder and modifies nothing else.

| Section | Question addressed | Output |
|---|---|---|
| 1 | Data, models, **persisted scores** | `scores_d1_seed42.npz` |
| 2 | Are the post-hoc maps injective as implemented? | `tab_injectivity.csv/.tex` |
| 3 | Threshold selection on the saved scores (Table 8) | `tab8_gridres.csv/.tex` |
| 4 | Sensitivity to the clip constant; strictly increasing rank map | `tab_eps_sensitivity.csv/.tex` |
| 5 | Is Δ unchanged after calibration? | `tab_delta_after_calibration.csv` |
| 6 | Counterexample of Remark 3.3; simulation | `tab_counterexample.csv` |
| 7 | Screening-flag behaviour with explicit case labels (Table 10) | `tab10_*.csv/.tex` |
| 8 | The two cost regimes as separate scenarios (Table 9) | `tab9_costregimes.csv/.tex` |
| 9 | Export of the LaTeX tables and of `revision_values.json` | — |
| 10 | Automated consistency checks | printed report |

**Run the cells in order, from top to bottom.** Only the configuration cell in Section 0 needs editing.
The software environment is printed by that cell and is the one recorded in Table 4 of the paper.

## 0 — Configuration (**the only cell to edit**)

In [ ]:
from pathlib import Path
from datetime import datetime
import os, sys, json, platform, warnings
warnings.filterwarnings("ignore")

# ------------------------------------------------------------------ EDIT ME
CODE_DIR = Path(".")        # folder holding c3e_framework.py and c3e_contribution3.py
CREDITCARD_CSV = r"path/to/creditcard.csv"   # Kaggle creditcard.csv (see Data availability)
ALLSEEDS_CSV   = None       # None -> locate c3e_all_seeds_raw.csv automatically
# --------------------------------------------------------------------------

SEED     = 42               # single seed for the whole D1 analysis (controlled comparison)
CFN, CFP = 10.0, 1.0        # fixed-cost regime of the paper
TAU_BAYES = CFP / (CFP + CFN)
N_GRID = 1001               # uniform threshold grid of the protocol
DELTA_STAR_LIST = [0.10, 0.15, 0.20, 0.25, 0.30]
KAPPA_LIST = [1.5, 2.0, 5.0]                  # excess-cost thresholds used to label Table 10
KAPPA_MAIN = 2.0
EPS_LIST = [1e-6, 1e-7, 1e-9, 1e-12, 1e-15]   # 1e-7 is the value used by the released code

NEEDED = ["c3e_framework.py", "c3e_contribution3.py"]

def _find_code_dir(start):
    """Locate a directory containing both modules; return None if not found."""
    if all((Path(start) / f).exists() for f in NEEDED):
        return Path(start)
    roots, seen = [Path.cwd(), Path.home() / "Desktop", Path.home()], set()
    for r in roots:
        if not r.exists() or r in seen:
            continue
        seen.add(r)
        try:
            for p in r.rglob("c3e_framework.py"):
                if (p.parent / "c3e_contribution3.py").exists():
                    return p.parent
        except (PermissionError, OSError):
            continue
    return None

def _find_file(name, explicit=None):
    """Locate a file; return the most recent match, or None."""
    if explicit and Path(explicit).exists():
        return Path(explicit)
    hits = []
    for r in [Path.cwd(), Path.home() / "Desktop", CODE_DIR.parent if CODE_DIR else Path.cwd()]:
        if not r or not Path(r).exists():
            continue
        try:
            hits += list(Path(r).rglob(name))
        except (PermissionError, OSError):
            continue
    return max(hits, key=lambda p: p.stat().st_mtime) if hits else None

found = _find_code_dir(CODE_DIR)
if found and found != CODE_DIR:
    print(f"[info] modules found elsewhere: {found}")
if found:
    CODE_DIR = found
sys.path.insert(0, str(CODE_DIR))

ALLSEEDS_CSV = _find_file("c3e_all_seeds_raw.csv", ALLSEEDS_CSV)

REV_DIR = Path("audit_output") / datetime.now().strftime("%Y-%m-%d_%H%M%S")
(REV_DIR / "tables").mkdir(parents=True, exist_ok=True)
(REV_DIR / "results").mkdir(parents=True, exist_ok=True)
TABLES, RESULTS = REV_DIR / "tables", REV_DIR / "results"
VALUES = {}   # values quoted in the paper -> computed value

print("Output folder   :", REV_DIR.resolve())
print("CODE_DIR        :", CODE_DIR)
for f in NEEDED:
    print(f"   {'OK     ' if (CODE_DIR/f).exists() else 'MISSING'}  {f}")
print("creditcard.csv  :", Path(CREDITCARD_CSV).exists(), "|", CREDITCARD_CSV)
print("all-seeds CSV   :", ALLSEEDS_CSV if ALLSEEDS_CSV else
      "NOT FOUND -> Section 7 (Table 10) will be skipped)")
# --- software environment: reported in Table 4 of the paper -------------------
print("\nEnvironment (Table 4):")
print(f"  python       {platform.python_version()}   {sys.executable}")
for _m in ("lightgbm", "xgboost", "catboost", "sklearn", "numpy", "pandas", "scipy"):
    try:
        print(f"  {_m:12s} {__import__(_m).__version__}")
    except Exception as _e:
        print(f"  {_m:12s} not installed ({_e})")
if not all((CODE_DIR/f).exists() for f in NEEDED):
    print("\n[!] Fix CODE_DIR above before continuing.")

## 1 — Dataset D1, models, and score persistence

Raw validation and test scores are written to disk. This is the condition under which every
subsequent section operates: no model is retrained after this cell, and every threshold reported in
the paper can be re-derived from this file.

The preprocessing and the trainers are those of `cf.run_experiment`, which produced Tables 5 and 7,
so the values below are directly comparable with them. The check printed at the end of the cell
compares the recomputed Δ with the values of Table 5.

In [ ]:
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

import c3e_framework as cf
import c3e_contribution3 as c3

# global seed (same convention as the multi-seed cell of the master notebook)
cf.SEED = SEED
try: c3.SEED = SEED
except Exception: pass
np.random.seed(SEED); os.environ["PYTHONHASHSEED"] = str(SEED)

# the registry holds a machine-specific path; override it here
cf.DATASET_REGISTRY["creditcard"].path = CREDITCARD_CSV

df_cc, spec_cc = cf.load_dataset("creditcard")
df_tr, df_val, df_te = cf.chronological_split(df_cc, spec_cc.label_col)

# Preprocessing and trainers IDENTICAL to cf.run_experiment, which produced Tables 5
# and 7: prepare_splits casts to float32 before imputation and standardisation, and the
# trainers are those of c3e_framework. Any other combination yields a different LightGBM
# fit, hence different Delta values and costs.
splits = cf.prepare_splits(df_tr, df_val, df_te, spec_cc.label_col, spec_cc.time_col)
X_tr, y_tr   = splits["X_train"], splits["y_train"].astype(int)
X_val, y_val = splits["X_val"],   splits["y_val"].astype(int)
X_te,  y_te  = splits["X_test"],  splits["y_test"].astype(int)
amount_te  = df_te["Amount"].to_numpy(dtype=float)  if "Amount" in df_te.columns  else None
amount_val = df_val["Amount"].to_numpy(dtype=float) if "Amount" in df_val.columns else None

MODELS       = ["LR", "RF", "XGB", "LGBM"]          # core: Tables 8 and 10
EXTRA_MODELS = ["CatBoost", "IF-Hybrid"]            # extra: Table 9 (six rows)
scores = {}
for m in MODELS:
    print(f"  training {m} ...", end=" ", flush=True)
    scorer = cf.MODEL_TRAINERS[m](X_tr, y_tr)          # framework trainers
    scores[m] = dict(val=np.asarray(scorer(X_val), dtype=float),
                     test=np.asarray(scorer(X_te),  dtype=float))
    print(f"OK  (val: {np.unique(scores[m]['val']).size} distinct scores)")

for m in EXTRA_MODELS:                               # optional: continue if unavailable
    try:
        print(f"  training {m} ...", end=" ", flush=True)
        sc = c3.MODEL_TRAINERS_C3[m](X_tr, y_tr, X_val, y_val)   # not present in cf
        if sc is None: raise RuntimeError("trainer returned None")
        scores[m] = dict(val=np.asarray(sc(X_val), dtype=float),
                         test=np.asarray(sc(X_te),  dtype=float))
        print("OK")
    except Exception as e:
        print(f"SKIPPED ({e})")
ALL_MODELS = [m for m in MODELS + EXTRA_MODELS if m in scores]

npz = RESULTS / f"scores_d1_seed{SEED}.npz"
np.savez_compressed(
    npz, y_val=y_val, y_te=y_te,
    amount_te=(amount_te   if amount_te   is not None else np.array([])),
    amount_val=(amount_val if amount_val is not None else np.array([])),
    **{f"{m}_val":  scores[m]["val"]  for m in ALL_MODELS},
    **{f"{m}_test": scores[m]["test"] for m in ALL_MODELS})
print("\nScores saved to:", npz.resolve())
print(f"N_val={len(y_val):,} (rho={y_val.mean():.4%})   N_test={len(y_te):,} (rho={y_te.mean():.4%})")


# --- reproducibility check: these values must match Table 5 ------------------
REF_TABLE5 = {"LR": (0.202, 202), "RF": (0.186, 196), "XGB": (0.187, 191), "LGBM": (0.868, 1858)}
print("\nCheck against Table 5 (five-seed means there; seed 42 only here):")
print(f"  {'model':8s} {'Delta':>8s} {'(ref)':>8s}   {'diff':>7s}")
for m in MODELS:
    d = float(roc_auc_score(y_val, scores[m]["val"]) - average_precision_score(y_val, scores[m]["val"]))
    ref = REF_TABLE5[m][0]
    flag = "" if abs(d - ref) < 0.02 else "   <-- MISMATCH, check the pipeline"
    print(f"  {m:8s} {d:8.3f} {ref:8.3f}   {d-ref:+7.3f}{flag}")

## 2 — Are the post-hoc maps injective as implemented?

`TemperatureScaling.predict` and `BetaCalibration.predict` apply `np.clip(p, eps, 1-eps)` with
`eps = 1e-7` before taking logits. The composition is therefore non-decreasing but constant on
$[0,\varepsilon]$ and on $[1-\varepsilon,1]$, so it is not injective and the rank-invariance
proposition does not apply to it as stated.

This cell counts distinct values at each stage of the map and reports how many observations sit at
each clip bound.

In [ ]:
from scipy.special import expit

def stage_audit(p, eps=1e-7, T=1.0):
    """Distinct-value count at each stage of sigma(logit(clip(p))/T)."""
    p = np.asarray(p, dtype=float)
    num = np.clip(p, eps, 1 - eps)
    den = np.clip(1 - p, eps, 1 - eps)
    lg  = np.log(num / den)
    out = expit(lg / T)
    return dict(n_raw=int(np.unique(p).size),
                n_at_low=int((p <= eps).sum()), n_at_high=int((p >= 1 - eps).sum()),
                n_distinct_clipped=int(np.unique(num / den).size),
                n_logit=int(np.unique(lg).size),
                n_out=int(np.unique(out).size),
                out_min=float(out.min()), out_max=float(out.max()))

rows = []
fitted_T, fitted_beta = {}, {}
for m in MODELS:
    pv = scores[m]["val"]
    ts = c3.TemperatureScaling().fit(pv, y_val); fitted_T[m] = float(ts.T_)
    a = stage_audit(pv, eps=1e-7, T=ts.T_)
    a.update(model=m, map="TS", T=round(float(ts.T_), 4))
    rows.append(a)
    try:
        bc = c3.BetaCalibration().fit(pv, y_val)
        pb = np.asarray(bc.predict(pv), dtype=float)
        fitted_beta[m] = [float(x) for x in np.ravel(bc.params_)]
        rows.append(dict(model=m, map="Beta", T=np.nan,
                         n_raw=int(np.unique(pv).size),
                         n_at_low=int((pv <= 1e-7).sum()), n_at_high=int((pv >= 1 - 1e-7).sum()),
                         n_distinct_clipped=np.nan, n_logit=np.nan,
                         n_out=int(np.unique(pb).size),
                         out_min=float(pb.min()), out_max=float(pb.max())))
    except Exception as e:
        print(f"  [Beta] {m}: {e}")

inj = pd.DataFrame(rows)[["model", "map", "T", "n_raw", "n_at_low", "n_at_high",
                          "n_logit", "n_out", "out_min", "out_max"]]
inj.to_csv(RESULTS / "tab_injectivity.csv", index=False)
print(inj.to_string(index=False))

lg = inj[(inj["model"] == "LGBM") & (inj["map"] == "TS")].iloc[0]
VALUES.update({
    "lgbm_T_fitted":        round(float(lg["T"]), 3),
    "lgbm_n_raw_unique":    int(lg["n_raw"]),
    "lgbm_n_clip_low":      int(lg["n_at_low"]),
    "lgbm_n_clip_high":     int(lg["n_at_high"]),
    "lgbm_n_ts_unique":     int(lg["n_out"]),
    "lgbm_ts_min":          round(float(lg["out_min"]), 4),
    "lgbm_ts_max":          round(float(lg["out_max"]), 4),
})
print("\n--- finding ---")
print(f"LGBM: {lg['n_raw']} distinct raw scores -> {lg['n_out']} after TS (T={lg['T']}).")
print(f"      {int(lg['n_at_low'])} observations at the lower clip (p<=1e-7), "
      f"{int(lg['n_at_high'])} at the upper clip (p>=1-1e-7).")
print("      A STRICTLY increasing map would preserve the distinct-value count:")
print("      any reduction is caused by the clip, not by the division by T.")

## 3 — Threshold selection on the saved scores (Table 8)

For every model and every candidate grid we record the **validation cost** — the objective actually
minimised — the cost of the frozen threshold on the test split, the **number of tied validation
minimisers**, the spread of test costs across those ties, the number of test transactions the frozen
rule flags, and the oracle cost (best threshold chosen on the test scores themselves).

These are the quantities that separate two competing explanations of a cost change: a genuine gain in
grid resolution would show up as a strictly lower validation cost, whereas a selection effect leaves
the validation optimum untouched and moves only which minimiser is chosen.

In [ ]:
def cost_at(y, p, tau, cfn=CFN, cfp=CFP):
    yp = (p >= tau)
    return float(cfn * np.sum(yp[y == 1] == False) + cfp * np.sum(yp[y == 0] == True))

def cost_curve(y, p, grid, cfn=CFN, cfp=CFP):
    return np.array([cost_at(y, p, t, cfn, cfp) for t in grid], dtype=float)

def three_grids(p_val):
    return {"uniform":  np.linspace(0.0, 1.0, N_GRID),
            "unique":   np.unique(p_val),
            "quantile": np.unique(np.quantile(p_val, np.linspace(0.0, 1.0, N_GRID)))}

def select_and_audit(p_val, p_te, grid, label, model, calib):
    grid = np.unique(np.clip(grid, 0.0, 1.0))
    cv   = cost_curve(y_val, p_val, grid)
    cmin = cv.min()
    tied = np.flatnonzero(cv == cmin)                 # tied validation minimisers
    tau  = float(grid[tied[0]])                       # tie-break convention: first minimiser (np.argmin)
    te_tied = np.array([cost_at(y_te, p_te, float(grid[j])) for j in tied], dtype=float)
    # oracle: best threshold chosen on the test set itself (lower bound, not deployable)
    oracle_grid = np.unique(p_te)
    oracle = float(cost_curve(y_te, p_te, oracle_grid).min())
    n_flag = int(np.sum(p_te >= tau))          # 0 => the frozen rule flags NOTHING
    return dict(model=model, calibrator=calib, grid=label,
                n_val_unique=int(np.unique(p_val).size),
                tau_star=round(tau, 6),
                val_cost=round(cmin, 1),
                n_tied_val_optima=int(tied.size),
                n_flagged_test=n_flag,
                test_cost=round(cost_at(y_te, p_te, tau), 1),
                test_cost_tied_min=round(float(te_tied.min()), 1),
                test_cost_tied_max=round(float(te_tied.max()), 1),
                test_cost_oracle=round(oracle, 1))

rows = []
for m in MODELS:
    pv, pt = scores[m]["val"], scores[m]["test"]
    d = roc_auc_score(y_val, pv) - average_precision_score(y_val, pv)
    for gname, g in three_grids(pv).items():
        r = select_and_audit(pv, pt, g, gname, m, "none"); r["delta"] = round(float(d), 3)
        rows.append(r)

# --- LGBM + Temperature Scaling -------------------------------------------
ts = c3.TemperatureScaling().fit(scores["LGBM"]["val"], y_val)
pv_ts = np.asarray(ts.predict(scores["LGBM"]["val"]), dtype=float)
pt_ts = np.asarray(ts.predict(scores["LGBM"]["test"]), dtype=float)
d_ts  = roc_auc_score(y_val, pv_ts) - average_precision_score(y_val, pv_ts)
for gname, g in three_grids(pv_ts).items():
    r = select_and_audit(pv_ts, pt_ts, g, gname, "LGBM", "temperature")
    r["delta"] = round(float(d_ts), 3)
    rows.append(r)

NULL_COST_TEST = CFN * int(y_te.sum())      # null-model cost: flag nothing
NULL_COST_VAL  = CFN * int(y_val.sum())
tab8 = pd.DataFrame(rows)[["model", "calibrator", "grid", "delta", "n_val_unique", "tau_star",
                           "val_cost", "n_tied_val_optima", "n_flagged_test", "test_cost",
                           "test_cost_tied_min", "test_cost_tied_max", "test_cost_oracle"]]
tab8.to_csv(RESULTS / "tab8_gridres.csv", index=False)
pd.set_option("display.width", 220)
print(tab8.to_string(index=False))
print(f"\nNull-model cost (flag nothing): validation {NULL_COST_VAL:,.0f} "
      f"({int(y_val.sum())} frauds) | test {NULL_COST_TEST:,.0f} ({int(y_te.sum())} frauds)")
deg = tab8[tab8.n_flagged_test == 0]
if len(deg):
    print("\n[!] Rows whose frozen threshold flags NO test transaction:")
    print(deg[["model", "calibrator", "grid", "tau_star", "test_cost"]].to_string(index=False))
    print("    Their cost equals the null-model cost exactly; this is not a deployable")
    print("    operating point.")

### 3b — Was the rule selected after calibration already reachable on the raw scores?

Because the map is non-decreasing, any rule $\{\tilde p \ge \tau\}$ coincides on the validation split
with a rule $\{p \ge c\}$ on the raw scores. This cell recovers that $c$, checks that the two rules
agree, compares the validation costs, and reports how many test transactions the frozen rule flags.
A rule that flags nothing costs exactly $C_{FN} \cdot N_1$, the null-model cost.

In [ ]:
pv_raw, pt_raw = scores["LGBM"]["val"], scores["LGBM"]["test"]

row_ts  = tab8[(tab8.model=="LGBM") & (tab8.calibrator=="temperature") & (tab8.grid=="uniform")].iloc[0]
row_unq = tab8[(tab8.model=="LGBM") & (tab8.calibrator=="none")        & (tab8.grid=="unique")].iloc[0]
row_uni = tab8[(tab8.model=="LGBM") & (tab8.calibrator=="none")        & (tab8.grid=="uniform")].iloc[0]
tau_ts  = float(row_ts.tau_star)

# c = smallest raw score whose TS image reaches tau_ts (TS is non-decreasing)
allraw = np.unique(np.concatenate([pv_raw, pt_raw]))
img    = np.asarray(ts.predict(allraw), dtype=float)
ok     = np.flatnonzero(img >= tau_ts)
c_back = float(allraw[ok[0]]) if ok.size else float("inf")

same_test = bool(np.array_equal(pt_ts >= tau_ts, pt_raw >= c_back))
same_val  = bool(np.array_equal(pv_ts >= tau_ts, pv_raw >= c_back))
uni_grid  = np.linspace(0.0, 1.0, N_GRID)
reachable_uniform = bool(any(np.array_equal(pv_raw >= g, pv_raw >= c_back) for g in uni_grid))
reachable_unique  = bool(any(np.array_equal(pv_raw >= g, pv_raw >= c_back) for g in np.unique(pv_raw)))

val_cost_ts_rule  = cost_at(y_val, pv_raw, c_back)
val_cost_raw_best = float(row_unq.val_cost)

print(f"tau* (TS, uniform grid)                    : {tau_ts:.6f}")
print(f"equivalent raw cut-point c                 : {c_back:.3e}")
print(f"same decision on validation / test         : {same_val} / {same_test}")
print(f"rule reachable by the raw uniform grid     : {reachable_uniform}")
print(f"rule reachable by the raw unique-score grid: {reachable_unique}")
print()
print(f"VALIDATION cost, raw unique-grid optimum   : {val_cost_raw_best:>10.1f}")
print(f"VALIDATION cost, rule selected after TS    : {val_cost_ts_rule:>10.1f}")
print(f"TEST cost, raw uniform / raw unique / TS   : "
      f"{row_uni.test_cost:.1f} / {row_unq.test_cost:.1f} / {row_ts.test_cost:.1f}")
print(f"tied validation minimisers (raw, unique)   : {int(row_unq.n_tied_val_optima)}"
      f"  -> associated test costs in [{row_unq.test_cost_tied_min:.1f}, {row_unq.test_cost_tied_max:.1f}]")
print()
n_flag_ts  = int(np.sum(pt_ts  >= tau_ts))
n_flag_raw = int(np.sum(pt_raw >= float(row_uni.tau_star)))
print(f"test transactions flagged: raw {n_flag_raw:,} | after TS {n_flag_ts:,}")
print(f"null-model cost (test)   : {NULL_COST_TEST:,.0f} "
      f"({int(y_te.sum())} frauds x C_FN={CFN:g})")

if n_flag_ts == 0:
    verdict = ("EMPTY RULE: after the map the frozen threshold exceeds the largest score, so "
               "the model flags nothing. The resulting cost is exactly the null-model cost. "
               "This is not a restored threshold transferability.")
elif val_cost_ts_rule >= val_cost_raw_best:
    verdict = ("SELECTION: the map does not lower the validation objective; the test gain comes "
               "from which minimiser is selected and where the frozen threshold falls.")
else:
    verdict = ("RESOLUTION: the map strictly lowers the validation cost with a non-empty rule; "
               "the map improves the operating point.")
print("\nVERDICT:", verdict)

VALUES.update({
    "lgbm_tau_ts":            round(tau_ts, 4),
    "lgbm_c_backmapped":      f"{c_back:.2e}",
    "lgbm_valcost_raw_best":  round(val_cost_raw_best, 1),
    "lgbm_valcost_ts_rule":   round(float(val_cost_ts_rule), 1),
    "lgbm_testcost_raw":      round(float(row_uni.test_cost), 1),
    "lgbm_testcost_ts":       round(float(row_ts.test_cost), 1),
    "lgbm_testcost_oracle":   round(float(row_uni.test_cost_oracle), 1),
    "lgbm_n_tied_optima":     int(row_unq.n_tied_val_optima),
    "lgbm_tied_test_min":     round(float(row_unq.test_cost_tied_min), 1),
    "lgbm_tied_test_max":     round(float(row_unq.test_cost_tied_max), 1),
    "lgbm_rule_reachable_raw": "yes" if reachable_unique else "no",
    "lgbm_nflag_ts":     n_flag_ts,
    "lgbm_nflag_raw":    n_flag_raw,
    "lgbm_nullcost_test": round(float(NULL_COST_TEST), 1),
    "n_fraud_test":      int(y_te.sum()),
})

## 4 — Where does the cost change come from?

Two controls, both on the saved scores.

1. **Sensitivity to `eps`.** The clip constant is a numerical guard with no statistical content. If
   the test cost depends on it, the reported result is an implementation artefact.
2. **Rank map.** $\varphi$ sends the $m$ distinct validation scores to $\{(k-\tfrac12)/m\}$. This is a
   strictly increasing map that spreads the scores as widely as possible, so it preserves Δ exactly.
   If score redistribution is the mechanism, this map must reproduce the effect.

In [ ]:
class TSeps:
    """Temperature Scaling with configurable eps (identical to the released code at eps=1e-7)."""
    def __init__(self, eps=1e-7): self.eps, self.T_ = eps, 1.0
    def _logit(self, s):
        e = self.eps
        return np.log(np.clip(s, e, 1-e) / np.clip(1-s, e, 1-e))
    def fit(self, s, y):
        from scipy.optimize import minimize_scalar
        lg, e = self._logit(s), self.eps
        f = lambda lT: -float((y*np.log(np.clip(expit(lg/np.exp(lT)), e, 1-e)) +
                               (1-y)*np.log(1-np.clip(expit(lg/np.exp(lT)), e, 1-e))).mean())
        self.T_ = float(np.exp(minimize_scalar(f, bounds=(-3, 3), method="bounded").x)); return self
    def predict(self, s): return expit(self._logit(s) / self.T_)

def rank_map(p_val_ref):
    """STRICTLY increasing map: distinct validation scores -> (k-1/2)/m."""
    u = np.unique(p_val_ref); t = (np.arange(1, u.size+1) - 0.5) / u.size
    return lambda s: np.interp(np.asarray(s, dtype=float), u, t, left=0.0, right=1.0)

uni = np.linspace(0.0, 1.0, N_GRID)
rows = []
for e in EPS_LIST:
    tse = TSeps(eps=e).fit(pv_raw, y_val)
    a, b = np.asarray(tse.predict(pv_raw)), np.asarray(tse.predict(pt_raw))
    r = select_and_audit(a, b, uni, "uniform", "LGBM", f"TS(eps={e:.0e})")
    r.update(eps=e, T=round(float(tse.T_), 4), n_out_unique=int(np.unique(a).size))
    rows.append(r)

phi = rank_map(pv_raw)
a, b = phi(pv_raw), phi(pt_raw)
r = select_and_audit(a, b, uni, "uniform", "LGBM", "rank map (strictly increasing)")
r.update(eps=np.nan, T=np.nan, n_out_unique=int(np.unique(a).size)); rows.append(r)

r = select_and_audit(pv_raw, pt_raw, uni, "uniform", "LGBM", "none"); r.update(
    eps=np.nan, T=np.nan, n_out_unique=int(np.unique(pv_raw).size)); rows.append(r)

eps_tab = pd.DataFrame(rows)[["calibrator", "eps", "T", "n_out_unique", "tau_star",
                              "val_cost", "n_tied_val_optima", "n_flagged_test",
                              "test_cost", "test_cost_oracle"]]
eps_tab.to_csv(RESULTS / "tab_eps_sensitivity.csv", index=False)
print(eps_tab.to_string(index=False))

c_raw  = float(eps_tab[eps_tab.calibrator == "none"].test_cost.iloc[0])
c_rank = float(eps_tab[eps_tab.calibrator.str.startswith("rank")].test_cost.iloc[0])
c_e7   = float(eps_tab[eps_tab.calibrator == "TS(eps=1e-07)"].test_cost.iloc[0])
spread = eps_tab[eps_tab.calibrator.str.startswith("TS(")].test_cost
print("\n--- reading ---")
print(f"raw: {c_raw:.0f}   |   TS(eps=1e-7): {c_e7:.0f}   |   rank map: {c_rank:.0f}")
print(f"test-cost range over eps in [1e-15, 1e-6]: "
      f"[{spread.min():.0f}, {spread.max():.0f}]")
print(f"null-model cost (test): {NULL_COST_TEST:,.0f}")
if (eps_tab[eps_tab.calibrator != "none"].n_flagged_test == 0).all():
    print("Every calibrated variant selects the EMPTY RULE: the gain is not a better")
    print("operating point, it is the absence of an operating point.")
VALUES.update({
    "lgbm_cost_rankmap":  round(c_rank, 1),
    "lgbm_cost_eps_min":  round(float(spread.min()), 1),
    "lgbm_cost_eps_max":  round(float(spread.max()), 1),
})

## 5 — Δ after calibration

Merging score levels creates ties, which can move ROC-AUC and PR-AUC. This cell recomputes Δ on the
saved scores for the raw scores, Temperature Scaling, Beta Calibration and the rank map.

In [ ]:
def delta_of(p, y=None):
    y = y_val if y is None else y
    return float(roc_auc_score(y, p) - average_precision_score(y, p))

variants = {"raw": pv_raw, "TS (eps=1e-7)": pv_ts, "rank map": phi(pv_raw)}
try:
    bc = c3.BetaCalibration().fit(pv_raw, y_val)
    variants["Beta"] = np.asarray(bc.predict(pv_raw), dtype=float)
except Exception as ex:
    print("[Beta] unavailable:", ex)

rows = [dict(variant=k,
             n_unique=int(np.unique(v).size),
             roc_auc=round(float(roc_auc_score(y_val, v)), 4),
             pr_auc=round(float(average_precision_score(y_val, v)), 4),
             delta=round(delta_of(v), 4)) for k, v in variants.items()]
dtab = pd.DataFrame(rows)
dtab.to_csv(RESULTS / "tab_delta_after_calibration.csv", index=False)
print(dtab.to_string(index=False))

d_raw = float(dtab[dtab["variant"] == "raw"].delta.iloc[0])
d_ts_ = float(dtab[dtab["variant"].str.startswith("TS")].delta.iloc[0])
print(f"\nDelta raw = {d_raw:.4f} ; Delta after TS = {d_ts_:.4f} ; difference = {abs(d_raw-d_ts_):.4f}")
print("Exact rank invariance is expected ONLY for a strictly increasing map")
print("(the rank-map row). Any non-zero difference for TS measures the clip effect.")
VALUES.update({"delta_lgbm_raw": round(d_raw, 3), "delta_lgbm_ts": round(d_ts_, 3),
               "delta_lgbm_rank": round(float(dtab[dtab["variant"] == "rank map"].delta.iloc[0]), 3)})

## 6 — The counterexample of Remark 3.3, and the reachable cut set

Remark 3.3 uses three scores $0.48, 0.49, 0.51$ (two negatives, one positive) and a grid of spacing
$\eta = 0.5$. This cell verifies both halves of the example, then measures by simulation how often a
strictly increasing map **degrades** the cost attainable on a fixed grid.

The statement of Proposition 3.3 is independent of any support width: if
$s_{(1)}<\dots<s_{(m)}$ are the distinct validation scores, the decision rules reachable by the grid
$\mathcal{T}$ are indexed by
$K(f,\mathcal{T}) = \{k : (s_{(k)}, s_{(k+1)}] \cap \mathcal{T} \neq \emptyset\}$.
A strictly increasing map leaves the attainable frontier unchanged but moves the gaps relative to the
grid, so $K$ may grow or shrink.

In [ ]:
def reachable_cuts(p, grid):
    """Number of distinct decision rules reachable by the grid."""
    return len({tuple(np.asarray(p) >= g) for g in grid})

# ---- counterexample of Remark 3.3 ----------------------------------------
y_ce  = np.array([0, 0, 1]); p_ce = np.array([0.48, 0.49, 0.51]); q_ce = np.array([0.1, 0.6, 0.9])
g_ce  = np.array([0.0, 0.5, 1.0]); eta = 0.5
def best_on(y, p, grid):
    c = np.array([CFN*np.sum((p < t)[y == 1]) + CFP*np.sum((p >= t)[y == 0]) for t in grid], float)
    return float(grid[int(np.argmin(c))]), float(c.min())

t1, c1 = best_on(y_ce, p_ce, g_ce); t2, c2 = best_on(y_ce, q_ce, g_ce)
ce = pd.DataFrame([
    dict(scores="0.48 / 0.49 / 0.51", width=round(np.ptp(p_ce), 2), rule=f"width < eta={eta}",
         tau=t1, cost=c1, reachable_rules=reachable_cuts(p_ce, g_ce)),
    dict(scores="0.10 / 0.60 / 0.90", width=round(np.ptp(q_ce), 2), rule=f"width >= eta={eta}",
         tau=t2, cost=c2, reachable_rules=reachable_cuts(q_ce, g_ce))])
print(ce.to_string(index=False))
print(f"\nThe support is narrower than the grid spacing, yet the grid separates the scores "
      f"perfectly (cost {c1:.0f}).")
print(f"A strictly increasing map widens the support, yet the attained cost rises "
      f"from {c1:.0f} to {c2:.0f}.")
ce.to_csv(RESULTS / "tab_counterexample.csv", index=False)

# ---- how often a strictly increasing map degrades the attained cost --------
rng = np.random.default_rng(SEED)
n_worse = n_better = n_equal = 0; N_SIM = 2000
for _ in range(N_SIM):
    n = int(rng.integers(20, 60)); y = (rng.random(n) < 0.2).astype(int)
    p = np.sort(rng.random(n) * 0.05 + 0.47)                    # narrow support
    k = float(rng.uniform(2.0, 12.0))
    q = 1.0 / (1.0 + np.exp(-k * (p - p.mean())))               # strictly increasing phi
    G = np.linspace(0, 1, 11)                                   # coarse grid (eta = 0.1)
    _, cp = best_on(y, p, G); _, cq = best_on(y, q, G)
    n_worse += cq > cp; n_better += cq < cp; n_equal += cq == cp
print(f"\nSimulation ({N_SIM} draws, grid spacing 0.1, strictly increasing phi):")
print(f"  validation cost degraded : {100*n_worse/N_SIM:5.1f} %")
print(f"  improved                 : {100*n_better/N_SIM:5.1f} %")
print(f"  unchanged                : {100*n_equal/N_SIM:5.1f} %")
print("A one-sided guarantee is therefore untenable: the effect goes both ways.")
VALUES.update({"ce_cost_raw": int(c1), "ce_cost_phi": int(c2),
               "sim_pct_worse": round(100*n_worse/N_SIM, 1),
               "sim_pct_better": round(100*n_better/N_SIM, 1)})

## 7 — Screening-flag behaviour with explicit case labels (Table 10)

Table 10 enumerates the screening outcome with a declared labelling rule:

> a (dataset, model) pair is **costly** when its frozen-threshold test cost exceeds $\kappa$ times the
> lowest cost obtained on the same dataset, and **regular** otherwise.

$\kappa$ is reported, its sensitivity is given, and for each $\delta^\dagger$ the costly pairs that go
unflagged and the regular pairs that are flagged are counted and named.

In [ ]:
src = Path(ALLSEEDS_CSV) if ALLSEEDS_CSV else None
if src is None or not src.exists():
    print("[!] c3e_all_seeds_raw.csv not found.\n"
          "    It is in results/ of the release (written by cell 2b of the master notebook).\n"
          "    Copy it next to this notebook, or set ALLSEEDS_CSV in Section 0,\n"
          "    then rerun this cell. The other sections are not affected.")
    pairs = None
else:
    raw = pd.read_csv(src)
    EXCLUDED = ["DNN", "DNN-Platt", "NODE", "TabNet"]    # not present in Table 5
    n0 = raw[["dataset", "model"]].drop_duplicates().shape[0]
    raw = raw[~raw["model"].isin(EXCLUDED)].copy()
    n1 = raw[["dataset", "model"]].drop_duplicates().shape[0]
    if n1 != n0:
        print(f"[info] {n0 - n1} pair(s) excluded ({', '.join(EXCLUDED)}): {n0} -> {n1} pairs.")
    pairs = (raw.groupby(["dataset", "model"], as_index=False)
                .agg(delta=("delta", "mean"), cost=("expected_cost", "mean"),
                     tau_star=("tau_star", "mean"), n_seeds=("delta", "size")))
    pairs["best_cost_ds"] = pairs.groupby("dataset")["cost"].transform("min")
    pairs["cost_ratio"]   = pairs["cost"] / pairs["best_cost_ds"]
    pairs["case"]         = np.where(pairs["cost_ratio"] > KAPPA_MAIN, "costly", "regular")
    pairs = pairs.sort_values(["dataset", "model"]).reset_index(drop=True)
    pairs.to_csv(RESULTS / "tab10_pairs_labelled.csv", index=False)
    if len(pairs) != 26:
        print(f"[!] {len(pairs)} pairs instead of the 26 of Table 5: check the all-seeds CSV.")
    print(f"{len(pairs)} pairs; costly (kappa={KAPPA_MAIN}): "
          f"{(pairs.case=='costly').sum()}; regular: {(pairs.case=='regular').sum()}\n")
    print(pairs[["dataset","model","delta","cost","cost_ratio","case"]]
          .to_string(index=False, float_format=lambda v: f"{v:,.3f}"))

In [ ]:
if pairs is not None:
    rows = []
    for k in KAPPA_LIST:
        case = np.where(pairs["cost_ratio"] > k, "costly", "regular")
        n_c, n_r = int((case == "costly").sum()), int((case == "regular").sum())
        for d in DELTA_STAR_LIST:
            flag = pairs["delta"].to_numpy() > d
            missed = pairs.loc[(case == "costly") & (~flag), ["dataset", "model"]]
            needless = pairs.loc[(case == "regular") & (flag), ["dataset", "model"]]
            rows.append(dict(
                kappa=k, delta_star=d,
                n_flagged=int(flag.sum()), n_total=len(pairs),
                n_costly=n_c, n_regular=n_r,
                n_missed=len(missed), n_needless=len(needless),
                missed="; ".join(f"{r.dataset}/{r.model}" for r in missed.itertuples()) or "-",
                needless_n=len(needless)))
    scr = pd.DataFrame(rows)
    scr.to_csv(RESULTS / "tab10_screening.csv", index=False)
    print(scr[scr.kappa == KAPPA_MAIN].to_string(index=False))
    print("\n(sensitivity to kappa)")
    print(scr[["kappa","delta_star","n_flagged","n_costly","n_missed","n_needless"]].to_string(index=False))

    main = scr[scr.kappa == KAPPA_MAIN]
    VALUES.update({
        "tab10_kappa":     KAPPA_MAIN,
        "tab10_n_costly":  int(main.n_costly.iloc[0]),
        "tab10_n_regular": int(main.n_regular.iloc[0]),
        "tab10_missed_020":   int(main[main.delta_star == 0.20].n_missed.iloc[0]),
        "tab10_needless_020": int(main[main.delta_star == 0.20].n_needless.iloc[0]),
        "tab10_missed_030":   int(main[main.delta_star == 0.30].n_missed.iloc[0]),
        "tab10_missed_list_030": main[main.delta_star == 0.30].missed.iloc[0],
        "tab10_flagged_020":  int(main[main.delta_star == 0.20].n_flagged.iloc[0]),
    })
    for d in DELTA_STAR_LIST:                      # values of Table 10
        r = main[main.delta_star == d].iloc[0]; k = f"{int(round(d*100)):03d}"
        VALUES[f"t10_flagged_{k}"]    = int(r.n_flagged)
        VALUES[f"t10_missed_{k}"]     = int(r.n_missed)
        VALUES[f"t10_needless_{k}"]   = int(r.n_needless)
        VALUES[f"t10_missedlist_{k}"] = r.missed

    # is the PaySim/LGBM pair flagged at 0.30?
    ps = pairs[(pairs.dataset.str.lower().str.contains("paysim")) & (pairs.model == "LGBM")]
    if len(ps):
        VALUES["paysim_lgbm_delta"] = round(float(ps.delta.iloc[0]), 3)
        print(f"\nPaySim/LGBM: Delta = {float(ps.delta.iloc[0]):.3f} -> flagged at 0.30? "
              f"{float(ps.delta.iloc[0]) > 0.30}")

## 8 — The two cost regimes as separate scenarios (Table 9)

The regimes do not use the same cost function: $C_{FN} = 10$ in one, $C_{FN}^{(i)} = \log(1 +
\text{Amount}_i)$ in the other, with $C_{FP} = 1$ unchanged. Comparing the totals therefore largely
compares the units. This cell prints the mean unit FN cost of each regime and reports the
**operating point** (FN and FP counts at the frozen threshold), which is the only quantity comparable
across regimes.

In [ ]:
assert amount_val is not None and amount_te is not None, "Amount column missing"
w_val, w_te = np.log1p(amount_val), np.log1p(amount_te)

def cost_fixed(y, p, tau):  return cost_at(y, p, tau)
def cost_log(y, p, tau, w): 
    yp = p >= tau
    return float(np.sum(w[(y == 1) & (~yp)]) + CFP * np.sum((y == 0) & yp))

uni = np.linspace(0.0, 1.0, N_GRID)
rows = []
for m in ALL_MODELS:
    pv, pt = scores[m]["val"], scores[m]["test"]
    tf = float(uni[int(np.argmin([cost_fixed(y_val, pv, t) for t in uni]))])
    tl = float(uni[int(np.argmin([cost_log(y_val, pv, t, w_val) for t in uni]))])
    fn_f = int(np.sum((y_te == 1) & (pt < tf))); fp_f = int(np.sum((y_te == 0) & (pt >= tf)))
    fn_l = int(np.sum((y_te == 1) & (pt < tl))); fp_l = int(np.sum((y_te == 0) & (pt >= tl)))
    rows.append(dict(model=m,
                     tau_fixed=round(tf, 4), FN_fixed=fn_f, FP_fixed=fp_f,
                     cost_fixed=round(cost_fixed(y_te, pt, tf), 2),
                     tau_log=round(tl, 4), FN_log=fn_l, FP_log=fp_l,
                     cost_log=round(cost_log(y_te, pt, tl, w_te), 2)))
tab9 = pd.DataFrame(rows)
tab9.to_csv(RESULTS / "tab9_costregimes.csv", index=False)
print(tab9.to_string(index=False))

unit_log = float(w_te[y_te == 1].mean())
print(f"\nUnit FN cost: fixed regime = {CFN:.1f}; log regime = {unit_log:.2f} on average "
      f"over test frauds (median {float(np.median(w_te[y_te==1])):.2f}).")
print(f"unit ratio = {CFN/unit_log:.2f}x (fixed-regime unit cost over mean log-regime unit cost).")
print("The totals therefore do not measure a saving: they change unit.")
VALUES.update({"unit_fn_log_mean": round(unit_log, 2),
               "unit_scale_ratio": round(CFN/unit_log, 2)})

## 9 — Export: LaTeX tables and `revision_values.json`

The `.tex` files are ready to be included or copied. The JSON holds every value that the
paper quotes from this notebook, and the cell prints them.

In [ ]:
# --- write the JSON first: it holds the values quoted in the paper ----------
(RESULTS / "revision_values.json").write_text(
    json.dumps(VALUES, indent=2, ensure_ascii=False), encoding="utf-8")
print("Written:", (RESULTS / "revision_values.json").resolve())

def _tex_cell(v, spec="{:,.3f}"):
    if v is None or (not isinstance(v, str) and pd.isna(v)):
        return "--"
    if isinstance(v, (bool, np.bool_)):
        return "yes" if v else "no"
    if isinstance(v, (int, np.integer)):
        return f"{int(v):,}".replace(",", "{,}")
    if isinstance(v, (float, np.floating)):
        return spec.format(float(v)).replace(",", "{,}")
    return str(v).replace("_", r"\_").replace("&", r"\&").replace("%", r"\%")

def to_tex(df, caption, label, cols=None, headers=None, fmt=None, star=True):
    d = df[cols] if cols else df
    fmt = fmt or {}
    headers = headers or list(d.columns)
    align = "".join("r" if pd.api.types.is_numeric_dtype(d[c]) else "l" for c in d.columns)
    env = "table*" if star else "table"
    out = [f"\\begin{{{env}}}", "\\centering", f"\\caption{{{caption}}}", f"\\label{{{label}}}",
           f"\\begin{{tabular}}{{{align}}}", "\\toprule",
           " & ".join(str(h).replace("_", r"\_") for h in headers) + r" \\", "\\midrule"]
    for _, row in d.iterrows():
        out.append(" & ".join(_tex_cell(row[c], fmt.get(c, "{:,.3f}")) for c in d.columns) + r" \\")
    out += ["\\bottomrule", "\\end{tabular}", f"\\end{{{env}}}"]
    return "\n".join(out)

F2 = {c: "{:,.1f}" for c in ["val_cost", "test_cost", "test_cost_oracle", "test_cost_tied_min",
                             "test_cost_tied_max", "cost_fixed", "cost_log"]}
F2.update({"tau_star": "{:.4f}", "delta": "{:.3f}", "tau_fixed": "{:.4f}",
           "tau_log": "{:.4f}", "T": "{:.3f}", "out_min": "{:.4f}", "out_max": "{:.4f}"})

exports = [
    ("tab8_gridres.tex", tab8, F2,
     "Threshold-grid comparison on D1 (creditcard), fixed-cost regime "
     "($C_{\\text{FN}}=10$, $C_{\\text{FP}}=1$), single seed 42, selection on saved raw scores. "
     "Val.\\ cost is the objective actually minimised; ties counts the validation-cost "
     "minimisers; Flagged is the number of test transactions above the frozen threshold "
     "(zero means the rule flags nothing); Oracle is the lowest test cost attainable by any "
     "threshold on the test scores.", "tab:gridres", True),
    ("tab9_costregimes.tex", tab9, F2,
     "Two cost scenarios on D1 (creditcard), no calibration. The regimes use different cost "
     "functions, so their totals are not commensurable; the comparable quantities are the "
     "operating points (FN, FP) at the frozen threshold.", "tab:tdcost", True),
    ("tab_injectivity.tex", inj, F2,
     "Distinct-value audit of the post-hoc maps on D1 validation scores. A strictly increasing "
     "map preserves the number of distinct values; any reduction is produced by the "
     "$\\varepsilon$-clip in the implementation.", "tab:injectivity", True),
    ("tab_eps_sensitivity.tex", eps_tab, F2,
     "Sensitivity of the LightGBM result to the clip constant $\\varepsilon$, and comparison "
     "with a strictly increasing rank map.", "tab:epssens", True),
]
if pairs is not None:
    exports += [
        ("tab10_screening.tex",
         scr[scr.kappa == KAPPA_MAIN][["delta_star", "n_flagged", "n_total", "n_costly",
                                       "n_missed", "n_needless", "missed"]],
         {"delta_star": "{:.2f}"},
         f"Screening-flag behaviour on the {len(pairs)} dataset--model pairs. A pair is costly "
         f"when its frozen-threshold test cost exceeds $\\kappa={KAPPA_MAIN}$ times the best cost "
         f"on the same dataset; denominators are explicit and missed cases are named.",
         "tab:delta_sensitivity", True),
        ("tab10_pairs_labelled.tex",
         pairs[["dataset", "model", "delta", "cost", "cost_ratio", "case"]],
         {"delta": "{:.3f}", "cost": "{:,.0f}", "cost_ratio": "{:.2f}"},
         "Case labels behind Table~\\ref{tab:delta_sensitivity}: every pair, its seed-mean "
         "$\\Delta$, its cost, its ratio to the best cost on the same dataset, and its label.",
         "tab:pairs_labelled", True),
    ]

for fname, df, fmt, cap, lab, star in exports:
    try:
        (TABLES / fname).write_text(to_tex(df, cap, lab, fmt=fmt, star=star), encoding="utf-8")
        print("  OK   ", fname)
    except Exception as e:
        print(f"  FAILED {fname}: {type(e).__name__}: {e}")

print("\nFiles written to", REV_DIR.resolve(), "\n")
for f in sorted(list(TABLES.glob("*")) + list(RESULTS.glob("*"))):
    print("  ", f.relative_to(REV_DIR))
print("\n--- values quoted in the paper ---")
for k, v in VALUES.items():
    print(f"  {k}  ->  {v}")

## 10 — Automated consistency checks

Each check below fails loudly if a statement made in the paper is not supported by the
numbers just computed.

In [ ]:
checks, failures, skipped = [], [], []
V = VALUES.get

def check(name, cond, detail="", needs=()):
    missing = [k for k in needs if V(k) is None]
    if missing:
        skipped.append((name, missing)); return
    checks.append((name, bool(cond), detail))
    if not cond: failures.append(name)

check("The clip reduces the distinct-value count (LGBM)",
      (V("lgbm_n_ts_unique") or 0) < (V("lgbm_n_raw_unique") or 0),
      f"{V('lgbm_n_raw_unique')} -> {V('lgbm_n_ts_unique')}",
      needs=("lgbm_n_ts_unique", "lgbm_n_raw_unique"))
try:
    check("The rank map preserves the distinct-value count",
          int(np.unique(phi(pv_raw)).size) == int(np.unique(pv_raw).size))
except NameError:
    skipped.append(("The rank map preserves the distinct-value count", ["phi"]))
check("Delta is exactly invariant under the rank map",
      abs((V("delta_lgbm_rank") or 0) - (V("delta_lgbm_raw") or 0)) < 5e-4,
      f"{V('delta_lgbm_rank')} vs {V('delta_lgbm_raw')}",
      needs=("delta_lgbm_rank", "delta_lgbm_raw"))
check("The cost obtained after calibration equals the null-model cost",
      abs((V("lgbm_testcost_ts") or 0) - (V("lgbm_nullcost_test") or 0)) < 1e-6,
      f"TS={V('lgbm_testcost_ts')} vs null model={V('lgbm_nullcost_test')}",
      needs=("lgbm_testcost_ts", "lgbm_nullcost_test"))
check("The frozen rule after calibration flags no transaction",
      V("lgbm_nflag_ts") == 0, f"{V('lgbm_nflag_ts')} flagged",
      needs=("lgbm_nflag_ts",))
check("No non-empty rule on the raw scores beats the null model",
      (V("lgbm_testcost_oracle") or 0) >= (V("lgbm_nullcost_test") or 0),
      f"oracle={V('lgbm_testcost_oracle')} vs null={V('lgbm_nullcost_test')}",
      needs=("lgbm_testcost_oracle", "lgbm_nullcost_test"))
check("The counterexample of Remark 3.3 holds (a strictly increasing map raises the cost)",
      (V("ce_cost_phi") or 0) > (V("ce_cost_raw") or 0), needs=("ce_cost_phi", "ce_cost_raw"))
check("The fixed/log total ratio is explained by the unit ratio",
      abs((V("unit_scale_ratio") or 0) - 3.5) < 2.5, f"{V('unit_scale_ratio')}x",
      needs=("unit_scale_ratio",))
try:
    if pairs is not None:
        check("The Table 10 denominators sum to the number of pairs",
              (V("tab10_n_costly") or 0) + (V("tab10_n_regular") or 0) == len(pairs),
              needs=("tab10_n_costly", "tab10_n_regular"))
        check("Table 10 covers the 26 pairs of Table 5", len(pairs) == 26,
              f"{len(pairs)} pairs")
        check("20 of the 26 pairs are flagged at delta*=0.20 (Table 5)",
              V("t10_flagged_020") == 20, f"{V('t10_flagged_020')} flagged",
              needs=("t10_flagged_020",))
        check("PaySim/LGBM is flagged at delta*=0.30 (Table 10)",
              (V("paysim_lgbm_delta") or 0) > 0.30, str(V("paysim_lgbm_delta")),
              needs=("paysim_lgbm_delta",))
except NameError:
    skipped.append(("Table 10 checks", ["pairs"]))

w = max([len(c[0]) for c in checks] + [10])
for name, ok, det in checks:
    print(f"[{'PASS' if ok else 'FAIL'}] {name:<{w}}  {det}")
for name, keys in skipped:
    print(f"[skip] {name:<{w}}  (missing: {', '.join(keys)})")
print("\n" + ("All checks pass." if not failures
              else f"{len(failures)} check(s) failed: {failures}"))